In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

g = 9.81  # m/s²

In [ ]:
df = pd.read_csv('imuLogPendulumTest2.csv')
df['time'] = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M:%S.%f')
df['t_s']  = (df['time'] - df['time'].iloc[0]).dt.total_seconds()

# Convert accelerations from milli-g to m/s²
df['AccX'] = df['AccX'] / 1000 * g
df['AccY'] = df['AccY'] / 1000 * g
df['AccZ'] = df['AccZ'] / 1000 * g

fs = 1.0 / np.median(np.diff(df['t_s'].values))
print(f'Sample rate: {fs:.2f} Hz,  Duration: {df.t_s.iloc[-1]:.1f} s')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df['t_s'], df['AccX'], label='AccX', lw=0.8)
axes[0].plot(df['t_s'], df['AccY'], label='AccY', lw=0.8)
axes[0].plot(df['t_s'], df['AccZ'], label='AccZ', lw=0.8)
axes[0].set_ylabel('Acceleration (m/s²)')
axes[0].set_title('Accelerometer – time series')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

axes[1].plot(df['t_s'], df['GyrX'], label='GyrX', lw=0.8)
axes[1].plot(df['t_s'], df['GyrY'], label='GyrY', lw=0.8)
axes[1].plot(df['t_s'], df['GyrZ'], label='GyrZ', lw=0.8)
axes[1].set_ylabel('Angular rate (deg/s)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Gyroscope – time series')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── FFT window (seconds) — adjust to focus on the swinging portion ──
T_START = 50.0
T_END   = 110.0
# ────────────────────────────────────────────────────────────────────

seg = df[(df['t_s'] >= T_START) & (df['t_s'] <= T_END)]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, col, color in zip(axes,
                           ['AccX', 'AccY', 'AccZ'],
                           ['C0',   'C1',   'C2']):
    s = seg[col].values
    s = s - s.mean()                    # remove DC
    n = len(s)
    yf    = rfft(s * np.hanning(n))
    xf    = rfftfreq(n, 1.0 / fs)
    power = np.abs(yf)

    ax.plot(xf, power, color=color, lw=0.9)
    ax.set_ylabel('Amplitude')
    ax.set_title(f'FFT – {col}  (t = {T_START}–{T_END} s)')
    ax.set_xlim(0, 2)
    ax.grid(True, alpha=0.4)

    mask      = (xf > 0.1) & (xf < 2.0)
    peak_freq = xf[mask][np.argmax(power[mask])]
    T_peak    = 1.0 / peak_freq
    L_est     = g * (T_peak / (2 * np.pi))**2
    ax.axvline(peak_freq, color='red', linestyle='--', lw=1.2,
               label=f'Peak: {peak_freq:.3f} Hz  →  T = {T_peak:.3f} s,  L ≈ {L_est*100:.1f} cm')
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()